<a href="https://colab.research.google.com/github/DmitriyKolesnikM8O/MOEX-Scripts/blob/main/Adviser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""
MOEX Screener v2 — технический скринер акций Московской биржи для часовой торговли
====================================================================================
Что изменилось по сравнению с v1 (правки по реальным багам и фидбеку):
  - Убран расчёт размера позиции в рублях/штуках — он был не нужен и вводил в
    заблуждение. Оставлены только уровни: вход, стоп-лосс, тейк-профит, R:R.
  - ИСПРАВЛЕНО округление цен — раньше дешёвые тикеры (доли рубля, напр. TGKA)
    схлопывались в 0.01/0.00/0.01 из-за фиксированных 2 знаков после запятой.
    Теперь округление адаптивное под порядок цены.
  - ИСПРАВЛЕН источник ложных свечных паттернов — раньше использовалась
    самодельная наивная логика без порога на размер тела свечи и без проверки,
    что последняя свеча вообще ЗАКРЫЛАСЬ (последний бар в свежих данных может
    быть ещё в процессе формирования — это давало мусорную геометрию и ложные
    сигналы). Обе проблемы исправлены; свечные паттерны теперь считает
    библиотека pandas-ta-classic (62 паттерна, протестированы на соответствие
    TA-Lib) вместо ручного кода.
  - ДОБАВЛЕНО много новых индикаторов через pandas-ta-classic: Stochastic,
    CCI, Williams %R, OBV, SuperTrend, Aroon — в дополнение к своим ADX/VWAP.
  - ДОБАВЛЕНО подтверждение по старшему таймфрейму (дневной тренд): часовой
    сигнал против дневного тренда получает штраф к score, по тренду — бонус.
    Это отдельно посчитано из уже скачанных часовых данных (ресемпл в дневки),
    без дополнительных запросов к API.
  - ДОБАВЛЕНА относительная сила к индексу IMOEX — тикер, который растёт
    быстрее рынка, интереснее для лонга, чем тот, что просто ползёт вместе
    с общим ростом рынка (и наоборот для шорта).
  - ML-компонент остался, но его вес в score по-прежнему автоматически
    приглушается, если AUC модели близко к случайности (см. предупреждение).

ВАЖНО, ПРОЧТИТЕ ПЕРЕД ИСПОЛЬЗОВАНИЕМ:
  - Это НЕ финансовая рекомендация и не Грааль. Инструмент сужает список
    кандидатов и структурирует анализ. Решение и его последствия — на вас.
  - ROC-AUC модели в районе 0.5-0.58 — это ожидаемая реальность рынка, а не
    брак: устойчивый легко находимый паттерн по цене/объёму давно был бы
    вычищен крупными фондами. Не судите весь инструмент по одному числу ML —
    техническая часть (тренд, паттерны, дивергенции, мультитаймфрейм,
    относительная сила) работает независимо от ML и весит в score больше.
  - Автоматические паттерны (голова-плечи, треугольники и т.д.) — это
    эвристика по локальным экстремумам, а не ручная разметка трейдера.
    На часовиках такие паттерны шумнее, чем на дневках/неделях.
  - Фундаментал сознательно не включён — инструмент под внутридневную/
    часовую спекуляцию, а не под долгосрочные инвестиции.

Как запустить в Google Colab:
  1. Создайте новый notebook на colab.research.google.com
  2. Runtime → Change runtime type → Hardware accelerator → None (GPU тут
     не используется ничем в этом скрипте и не ускоряет ничего)
  3. Вставьте содержимое этого файла в одну ячейку и запустите
  4. Настройте параметры в блоке CONFIG ниже под себя
"""

import os
import time
import pickle
import warnings
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.signal import argrelextrema

warnings.filterwarnings("ignore")


def _ensure(pkg_import_name, pip_name=None):
    try:
        return __import__(pkg_import_name)
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", pip_name or pkg_import_name])
        return __import__(pkg_import_name)


_ensure("tqdm")
_ensure("sklearn", "scikit-learn")
try:
    import pandas_ta_classic as pta
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pandas-ta-classic"])
    import pandas_ta_classic as pta

from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# =============================================================================
# CONFIG — настройте под себя
# =============================================================================
CONFIG = {
    "history_days": 240,          # сколько дней истории тянуть для часовых свечей
    "candle_interval": 60,        # 60 = часовой таймфрейм (MOEX ISS interval)
    "max_tickers": None,          # None = использовать liquidity_top_n, либо число для теста
    "liquidity_top_n": 120,       # сколько самых ликвидных тикеров брать в работу
    "min_atr_pct": 0.15,          # мин. волатильность (ATR% от цены) — отсекает фонды
                                   # ликвидности (LQDT, AKMM, SBMM и т.п.)
    "atr_stop_mult": 1.5,         # множитель ATR для стоп-лосса
    "risk_reward_target": 2.0,    # целевое соотношение прибыль/риск для тейк-профита
    "ml_horizon_bars": 6,         # на сколько часовых баров вперёд прогнозируем движение
    "ml_up_threshold_atr": 0.5,   # рост считается "успешным", если цена ушла > 0.5*ATR
    "rel_strength_window": 20,    # окно (в часовых барах) для относительной силы к IMOEX
    "top_n_report": 15,           # сколько лучших идей показать в итоге
    "max_workers": 10,            # сколько тикеров качать параллельно
    "cache_dir": "/content/moex_cache",  # кэш на время сессии
    "cache_ttl_hours": 6,         # сколько часов кэш считается свежим
    "swing_order": 3,             # чувствительность поиска локальных экстремумов
    "pattern_lookback_bars": 90,  # окно поиска графических паттернов
}

ISS_BASE = "https://iss.moex.com/iss"

import re

# Русские названия для самых значимых свечных паттернов из pandas-ta-classic.
# Если паттерна нет в словаре — показываем его техническое имя как есть.
CDL_NAMES_RU = {
    "CDL_ENGULFING": "Поглощение",
    "CDL_HAMMER": "Молот",
    "CDL_HANGINGMAN": "Повешенный",
    "CDL_SHOOTINGSTAR": "Падающая звезда",
    "CDL_INVERTEDHAMMER": "Перевёрнутый молот",
    "CDL_DOJI": "Доджи",
    "CDL_DRAGONFLYDOJI": "Доджи-стрекоза",
    "CDL_GRAVESTONEDOJI": "Доджи-надгробие",
    "CDL_LONGLEGGEDDOJI": "Доджи (длинные тени)",
    "CDL_RICKSHAWMAN": "Доджи-рикша",
    "CDL_MORNINGSTAR": "Утренняя звезда",
    "CDL_MORNINGDOJISTAR": "Утренняя звезда-доджи",
    "CDL_EVENINGSTAR": "Вечерняя звезда",
    "CDL_EVENINGDOJISTAR": "Вечерняя звезда-доджи",
    "CDL_3WHITESOLDIERS": "Три белых солдата",
    "CDL_3BLACKCROWS": "Три чёрные вороны",
    "CDL_HARAMI": "Харами",
    "CDL_HARAMICROSS": "Харами-крест",
    "CDL_DARKCLOUDCOVER": "Завеса из тёмных облаков",
    "CDL_PIERCING": "Просвет в облаках",
    "CDL_MARUBOZU": "Марубозу",
    "CDL_CLOSINGMARUBOZU": "Марубозу закрытия",
    "CDL_SPINNINGTOP": "Волчок",
    "CDL_HIGHWAVE": "Высокая волна",
    "CDL_BELTHOLD": "Пояс-удержание",
    "CDL_ABANDONEDBABY": "Брошенный младенец",
    "CDL_TRISTAR": "Три звезды",
    "CDL_KICKING": "Пинок",
    "CDL_MATCHINGLOW": "Совпадающий минимум",
    "CDL_TAKURI": "Такури",
    "CDL_THRUSTING": "Толчок",
    "CDL_ONNECK": "На шее",
    "CDL_INNECK": "В шее",
    "CDL_STICKSANDWICH": "Сэндвич",
    "CDL_UPSIDEGAP2CROWS": "Разрыв вверх, две вороны",
    "CDL_HIKKAKE": "Хиккаке",
}


# =============================================================================
# ФОРМАТИРОВАНИЕ ЦЕН
# =============================================================================
def smart_round(x, sig_figs=4):
    """
    Адаптивное округление: у дорогих бумаг (SBER ~280) — 2 знака,
    у дешёвых (TGKA ~0.0075) — достаточно значащих цифр, чтобы стоп и
    тейк не схлопывались в одно и то же число.
    """
    if x is None or not np.isfinite(x):
        return x
    if x == 0:
        return 0.0
    if abs(x) >= 1:
        return round(x, 2)
    magnitude = int(np.floor(np.log10(abs(x))))
    decimals = min(max(-magnitude + (sig_figs - 1), 2), 8)
    return round(x, decimals)


# =============================================================================
# 1. ЗАГРУЗКА ДАННЫХ С MOEX ISS API
# =============================================================================
def get_liquid_tickers(top_n):
    """Один запрос отдаёт live-оборот сразу по всем акциям TQBR — быстрый префильтр."""
    url = f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,VALTODAY",
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    js = r.json()

    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])
    md["VALTODAY"] = pd.to_numeric(md["VALTODAY"], errors="coerce").fillna(0)

    merged = sec.merge(md, on="SECID", how="left").fillna({"VALTODAY": 0})
    if merged["VALTODAY"].sum() > 0:
        merged = merged.sort_values("VALTODAY", ascending=False)
    return merged["SECID"].head(top_n).tolist()


def _cache_path(cache_dir, key):
    return os.path.join(cache_dir, f"{key}.pkl")


def _load_from_cache(cache_dir, key, ttl_hours):
    path = _cache_path(cache_dir, key)
    if not os.path.exists(path):
        return None
    age_hours = (time.time() - os.path.getmtime(path)) / 3600
    if age_hours > ttl_hours:
        return None
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception:
        return None


def _save_to_cache(cache_dir, key, df):
    os.makedirs(cache_dir, exist_ok=True)
    with open(_cache_path(cache_dir, key), "wb") as f:
        pickle.dump(df, f)


def _fetch_candles_raw(url, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    all_rows, start, columns = [], 0, None
    while True:
        params = {
            "from": since.strftime("%Y-%m-%d"),
            "till": till.strftime("%Y-%m-%d"),
            "interval": interval,
            "start": start,
        }
        r = requests.get(url, params=params, timeout=20)
        if r.status_code != 200:
            break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None:
            columns = js.get("columns", [])
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 500:
            break
        start += len(rows)

    if not all_rows or columns is None:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={
        "open": "Open", "close": "Close", "high": "High",
        "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"
    })
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date")
    # MOEX ISS иногда отдаёт повторяющийся бар на стыке страниц пагинации —
    # без дедупликации это ломает любое дальнейшее выравнивание по датам
    df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)

    # ВАЖНО: если последняя свеча ещё не закрылась (её конец в будущем
    # относительно момента запроса), выкидываем её — иначе индикаторы и
    # свечные паттерны считаются по недостроенному бару и дают мусор.
    if len(df) > 0 and df["DateEnd"].iloc[-1] > pd.Timestamp.now():
        df = df.iloc[:-1]

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)


def get_hourly_candles(ticker, days_back, interval=60):
    url = (f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/"
           f"securities/{ticker}/candles.json")
    return _fetch_candles_raw(url, days_back, interval)


def get_index_candles(secid, days_back, interval=60):
    """Свечи по индексу (по умолчанию IMOEX) — для расчёта относительной силы."""
    url = f"{ISS_BASE}/engines/stock/markets/index/boards/SNDX/securities/{secid}/candles.json"
    return _fetch_candles_raw(url, days_back, interval)


def download_all_candles(tickers, days_back, interval, cache_dir, ttl_hours, max_workers):
    results = {}
    to_download = []
    for t in tickers:
        cached = _load_from_cache(cache_dir, t, ttl_hours)
        if cached is not None and not cached.empty:
            results[t] = cached
        else:
            to_download.append(t)

    if results:
        print(f"   Из кэша сессии загружено сразу: {len(results)} тикеров")

    if to_download:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(get_hourly_candles, t, days_back, interval): t
                       for t in to_download}
            for fut in tqdm(as_completed(futures), total=len(futures),
                             desc="   Качаю свечи параллельно"):
                t = futures[fut]
                try:
                    df = fut.result()
                    if not df.empty:
                        results[t] = df
                        _save_to_cache(cache_dir, t, df)
                except Exception:
                    pass

    return results


# =============================================================================
# 2. ИНДИКАТОРЫ
# =============================================================================
def add_indicators(df):
    df = df.copy()
    df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()

    delta = df["Close"].diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["RSI14"] = (100 - (100 / (1 + rs))).fillna(50)

    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = df["MACD"] - df["MACD_signal"]

    prev_close = df["Close"].shift(1)
    tr = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs()
    ], axis=1).max(axis=1)
    df["ATR14"] = tr.ewm(alpha=1/14, adjust=False).mean()
    df["ATR_pct"] = df["ATR14"] / df["Close"] * 100

    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    df["BB_upper"] = sma20 + 2 * std20
    df["BB_lower"] = sma20 - 2 * std20
    df["BB_pctB"] = (df["Close"] - df["BB_lower"]) / (df["BB_upper"] - df["BB_lower"])

    df["Vol_SMA20"] = df["Volume"].rolling(20).mean()
    df["Vol_ratio"] = df["Volume"] / df["Vol_SMA20"].replace(0, np.nan)

    df["trend_up"] = ((df["EMA20"] > df["EMA50"]) & (df["EMA50"] > df["EMA200"])).astype(int)
    df["trend_down"] = ((df["EMA20"] < df["EMA50"]) & (df["EMA50"] < df["EMA200"])).astype(int)

    up_move, down_move = df["High"].diff(), -df["Low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr_for_adx = df["ATR14"].replace(0, np.nan)
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/14, adjust=False).mean() / atr_for_adx
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/14, adjust=False).mean() / atr_for_adx
    dx = ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)) * 100
    df["ADX14"] = dx.ewm(alpha=1/14, adjust=False).mean().fillna(0)

    session = df["Date"].dt.date
    typical_price = (df["High"] + df["Low"] + df["Close"]) / 3
    pv = typical_price * df["Volume"]
    df["VWAP"] = pv.groupby(session).cumsum() / df["Volume"].groupby(session).cumsum().replace(0, np.nan)
    df["price_vs_vwap_pct"] = (df["Close"] - df["VWAP"]) / df["VWAP"] * 100

    # --- Доп. индикаторы из pandas-ta-classic (Stochastic, CCI, Williams %R,
    # OBV, SuperTrend, Aroon) — обёрнуто в try, чтобы падение одного
    # индикатора не роняло весь пайплайн по всем тикерам.
    ta_input = df.rename(columns={
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Volume": "volume"
    }).set_index("Date")

    def _reindexed(result):
        """pandas-ta иногда сам выкидывает NaN-строки из результата — выравниваем
        обратно по полному индексу дат, чтобы длина всегда совпадала с df."""
        if result is None:
            return None
        return result.reindex(ta_input.index)

    try:
        stoch = _reindexed(ta_input.ta.stoch(k=14, d=3, smooth_k=3))
        if stoch is not None and not stoch.empty:
            df["STOCH_k"] = stoch.iloc[:, 0].values
            df["STOCH_d"] = stoch.iloc[:, 1].values
    except Exception:
        df["STOCH_k"], df["STOCH_d"] = np.nan, np.nan

    try:
        cci = _reindexed(ta_input.ta.cci(length=14))
        df["CCI14"] = cci.values if cci is not None else np.nan
    except Exception:
        df["CCI14"] = np.nan

    try:
        willr = _reindexed(ta_input.ta.willr(length=14))
        df["WILLR14"] = willr.values if willr is not None else np.nan
    except Exception:
        df["WILLR14"] = np.nan

    try:
        obv = _reindexed(ta_input.ta.obv())
        df["OBV"] = obv.values if obv is not None else np.nan
        df["OBV_SMA20"] = pd.Series(df["OBV"]).rolling(20).mean().values
    except Exception:
        df["OBV"], df["OBV_SMA20"] = np.nan, np.nan

    try:
        aroon = _reindexed(ta_input.ta.aroon(length=14))
        if aroon is not None and not aroon.empty:
            df["AROON_up"] = aroon.iloc[:, 0].values
            df["AROON_down"] = aroon.iloc[:, 1].values
    except Exception:
        df["AROON_up"], df["AROON_down"] = np.nan, np.nan

    try:
        st = _reindexed(ta_input.ta.supertrend(length=7, multiplier=3.0))
        if st is not None and not st.empty:
            dir_col = [c for c in st.columns if c.startswith("SUPERTd")]
            df["SuperTrend_dir"] = st[dir_col[0]].values if dir_col else np.nan
    except Exception:
        df["SuperTrend_dir"] = np.nan

    # Защитный барьер: если какой-то из шагов выше случайно продублировал
    # колонку (встречалось на реальных данных с нетипичной формой ответа
    # pandas-ta для отдельных проблемных тикеров), не даём этому дальше
    # ломать сравнения DataFrame при обучении ML.
    df = df.loc[:, ~df.columns.duplicated()]
    # ВАЖНО: НЕ храним ta_input в df.attrs. pandas при pd.concat() пытается
    # сравнить .attrs всех объединяемых таблиц между собой (obj.attrs == attrs),
    # а сравнение словарей, где значение — целый DataFrame, ломается с
    # "ambiguous truth value" / "can only compare identically-labeled
    # DataFrame objects". Это и было причиной падения на этапе обучения ML —
    # нашёл через прямой traceback, не наугад.
    return df


def add_daily_confirmation(df):
    """
    Ресемплит уже скачанные часовые данные в дневные бары и считает дневной
    тренд (EMA20 vs EMA50 на дневках). Дополнительных запросов к API не нужно.
    Возвращает True/False/None (None = не хватает дневной истории).
    """
    daily = (df.set_index("Date")
               .resample("1D")
               .agg({"Open": "first", "High": "max", "Low": "min",
                     "Close": "last", "Volume": "sum"})
               .dropna())
    if len(daily) < 55:
        return None
    ema20 = daily["Close"].ewm(span=20, adjust=False).mean()
    ema50 = daily["Close"].ewm(span=50, adjust=False).mean()
    if ema20.iloc[-1] > ema50.iloc[-1]:
        return True
    elif ema20.iloc[-1] < ema50.iloc[-1]:
        return False
    return None


def compute_relative_strength(df, index_df, window):
    """
    Относительная сила к IMOEX: разница накопленной доходности тикера и
    индекса за последние `window` часовых баров, в процентных пунктах.
    Положительное значение = тикер обгоняет рынок.

    ВАЖНО: используем reindex/merge_asof по отсортированному индексу вместо
    обычного merge на Date. Обычный merge на дублирующихся датах (а MOEX
    ISS иногда отдаёт повторяющиеся бары на стыке страниц пагинации) даёт
    строк БОЛЬШЕ, чем в исходном df, и тогда результат перестаёт совпадать
    по длине с df — именно это рушило пайплайн на реальных данных.
    """
    if index_df is None or index_df.empty:
        df["rel_strength_pct"] = np.nan
        return df

    idx = index_df[["Date", "Close"]].rename(columns={"Close": "Index_Close"})
    idx = idx.drop_duplicates(subset="Date").sort_values("Date")
    base = df[["Date"]].sort_values("Date")

    aligned = pd.merge_asof(base, idx, on="Date", direction="backward")
    aligned = aligned.set_index(base.index).reindex(df.index)

    ticker_ret = df["Close"].pct_change(window)
    index_ret = aligned["Index_Close"].pct_change(window)
    df["rel_strength_pct"] = ((ticker_ret.values - index_ret.values) * 100)
    return df


# =============================================================================
# 3. СВЕЧНЫЕ ПАТТЕРНЫ (через pandas-ta-classic — надёжнее самодельных правил)
# =============================================================================
def detect_candle_patterns(df):
    """Возвращает список паттернов, сработавших на последней ЗАКРЫТОЙ свече."""
    if len(df) < 5:
        return []
    # Пересчитываем вспомогательную таблицу для pandas-ta локально (дёшево —
    # это просто переименование 5 колонок), а не берём из df.attrs — см.
    # комментарий в add_indicators про то, почему там хранить нельзя.
    ta_input = df.rename(columns={
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Volume": "volume"
    }).set_index("Date")
    try:
        cdl = ta_input.ta.cdl_pattern(name="all")
        if cdl is not None:
            cdl = cdl.reindex(ta_input.index)  # см. комментарий в add_indicators про reindex
    except Exception:
        return []
    if cdl is None or cdl.empty:
        return []

    last = cdl.iloc[-1]
    found = []
    for col, val in last.items():
        if val == 0 or pd.isna(val):
            continue
        base_name = re.sub(r"_\d+_[\d.]+$", "", col).upper()
        name_ru = CDL_NAMES_RU.get(base_name, col.replace("CDL_", "").replace("_", " ").title())
        direction = "бычий" if val > 0 else "медвежий"
        found.append(f"{name_ru} ({direction})")
    return found


# =============================================================================
# 4. ГРАФИЧЕСКИЕ ПАТТЕРНЫ (свинги) И ДИВЕРГЕНЦИИ
# =============================================================================
def find_swings(series, order):
    values = series.values
    highs_idx = argrelextrema(values, np.greater_equal, order=order)[0]
    lows_idx = argrelextrema(values, np.less_equal, order=order)[0]
    highs_idx = np.array(sorted(set(highs_idx.tolist())))
    lows_idx = np.array(sorted(set(lows_idx.tolist())))
    return highs_idx, lows_idx


def detect_chart_patterns(df, lookback, swing_order):
    found = []
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return found

    highs, lows = window["High"], window["Low"]
    close_last = window["Close"].iloc[-1]
    high_idx, _ = find_swings(highs, swing_order)
    _, low_idx = find_swings(lows, swing_order)
    tol = window["Close"].std() * 0.6 if window["Close"].std() > 0 else close_last * 0.01

    if len(high_idx) >= 3:
        last3 = high_idx[-3:]
        h_vals = highs.iloc[last3].values
        left_sh, head, right_sh = h_vals[0], h_vals[1], h_vals[2]
        if head > left_sh + tol and head > right_sh + tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = lows.iloc[last3[0]:last3[2]].min()
            found.append("Голова и плечи (подтверждено пробоем шеи вниз)" if close_last < neckline
                          else "Голова и плечи (формируется, шея не пробита)")

    if len(low_idx) >= 3:
        last3 = low_idx[-3:]
        l_vals = lows.iloc[last3].values
        left_sh, head, right_sh = l_vals[0], l_vals[1], l_vals[2]
        if head < left_sh - tol and head < right_sh - tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = highs.iloc[last3[0]:last3[2]].max()
            found.append("Перевёрнутая голова и плечи (подтверждено пробоем вверх)" if close_last > neckline
                          else "Перевёрнутая голова и плечи (формируется)")

    if len(high_idx) >= 2:
        h1, h2 = highs.iloc[high_idx[-2]], highs.iloc[high_idx[-1]]
        if abs(h1 - h2) < tol and (high_idx[-1] - high_idx[-2]) > swing_order * 2:
            trough = lows.iloc[high_idx[-2]:high_idx[-1]].min()
            found.append("Двойная вершина (подтверждена пробоем вниз)" if close_last < trough
                          else "Двойная вершина (формируется)")

    if len(low_idx) >= 2:
        l1, l2 = lows.iloc[low_idx[-2]], lows.iloc[low_idx[-1]]
        if abs(l1 - l2) < tol and (low_idx[-1] - low_idx[-2]) > swing_order * 2:
            peak = highs.iloc[low_idx[-2]:low_idx[-1]].max()
            found.append("Двойное дно (подтверждено пробоем вверх)" if close_last > peak
                          else "Двойное дно (формируется)")

    if len(high_idx) >= 3 and len(low_idx) >= 3:
        hx, hy = high_idx[-3:], highs.iloc[high_idx[-3:]].values
        lx, ly = low_idx[-3:], lows.iloc[low_idx[-3:]].values
        slope_high = np.polyfit(hx, hy, 1)[0]
        slope_low = np.polyfit(lx, ly, 1)[0]
        flat = close_last * 0.0006
        if abs(slope_high) < flat and slope_low > flat:
            found.append("Восходящий треугольник")
        elif slope_high < -flat and abs(slope_low) < flat:
            found.append("Нисходящий треугольник")
        elif slope_high < -flat and slope_low > flat:
            found.append("Симметричный треугольник (сужение, ждём пробоя)")

    range_high = window["High"].iloc[-lookback:-1].max()
    range_low = window["Low"].iloc[-lookback:-1].min()
    last_vol, avg_vol = window["Volume"].iloc[-1], window["Volume"].iloc[:-1].mean()
    if last_vol > avg_vol * 1.5:
        if close_last > range_high:
            found.append("Пробой диапазона вверх на объёме")
        elif close_last < range_low:
            found.append("Пробой диапазона вниз на объёме")

    return found


def detect_divergence(df, lookback, swing_order):
    found = []
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return found

    _, low_idx = find_swings(window["Low"], swing_order)
    high_idx, _ = find_swings(window["High"], swing_order)

    if len(low_idx) >= 2:
        i1, i2 = low_idx[-2], low_idx[-1]
        if (window["Low"].iloc[i2] < window["Low"].iloc[i1] and
                (window["RSI14"].iloc[i2] > window["RSI14"].iloc[i1] or
                 window["MACD_hist"].iloc[i2] > window["MACD_hist"].iloc[i1])):
            found.append("Бычья дивергенция (цена ниже, осциллятор выше)")

    if len(high_idx) >= 2:
        i1, i2 = high_idx[-2], high_idx[-1]
        if (window["High"].iloc[i2] > window["High"].iloc[i1] and
                (window["RSI14"].iloc[i2] < window["RSI14"].iloc[i1] or
                 window["MACD_hist"].iloc[i2] < window["MACD_hist"].iloc[i1])):
            found.append("Медвежья дивергенция (цена выше, осциллятор ниже)")

    return found


# =============================================================================
# 5. ML-МОДЕЛЬ
# =============================================================================
FEATURES = ["RSI14", "MACD_hist", "ATR_pct", "BB_pctB", "Vol_ratio",
            "trend_up", "trend_down", "ADX14", "price_vs_vwap_pct",
            "STOCH_k", "CCI14", "WILLR14", "rel_strength_pct"]


def build_ml_dataset(df, horizon, up_thresh_atr):
    # Защитный барьер от дублирующихся колонок (см. комментарий в add_indicators) —
    # без него операции ниже могут внезапно начать сравнивать DataFrame с DataFrame
    # вместо Series с Series и падать с "identically-labeled" ошибкой.
    d = df.loc[:, ~df.columns.duplicated()].copy()
    future_close = d["Close"].shift(-horizon)
    move = future_close - d["Close"]
    d["target"] = (move > up_thresh_atr * d["ATR14"]).astype(int)
    d = d.dropna(subset=FEATURES + ["target"])
    return d


def train_and_predict(all_data_by_ticker, horizon, up_thresh_atr):
    frames = []
    failed_tickers = []
    for ticker, df in all_data_by_ticker.items():
        # Один "проблемный" тикер (нетипичная форма данных, редкий edge-case
        # от pandas-ta и т.п.) не должен ронять обучение модели по всем
        # остальным — ловим и пропускаем, а не падаем всем пайплайном.
        try:
            ds = build_ml_dataset(df, horizon, up_thresh_atr)
            if len(ds) > 50:
                ds = ds.copy()
                ds["ticker"] = ticker
                frames.append(ds)
        except Exception as e:
            failed_tickers.append((ticker, str(e)))

    if failed_tickers:
        print(f"   Пропущено при построении ML-датасета: {len(failed_tickers)} тикеров "
              f"(напр. {failed_tickers[0][0]}: {failed_tickers[0][1][:80]})")

    if not frames:
        return None, None

    full = pd.concat(frames, ignore_index=True).sort_values("Date")
    split_idx = int(len(full) * 0.8)
    train, valid = full.iloc[:split_idx], full.iloc[split_idx:]

    model = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=50,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
    model.fit(train[FEATURES], train["target"])

    auc = None
    if valid["target"].nunique() > 1:
        preds = model.predict_proba(valid[FEATURES])[:, 1]
        auc = roc_auc_score(valid["target"], preds)
    return model, auc


# =============================================================================
# 6. РИСК-МЕНЕДЖМЕНТ (только уровни — без расчёта размера позиции)
# =============================================================================
def compute_risk_levels(last_row, direction, atr_mult, rr_target):
    price, atr = last_row["Close"], last_row["ATR14"]
    min_risk = price * 0.001

    if direction == "long":
        stop = price - atr_mult * atr
        risk_per_unit = max(price - stop, min_risk)
        target = price + risk_per_unit * rr_target
    else:
        stop = price + atr_mult * atr
        risk_per_unit = max(stop - price, min_risk)
        target = price - risk_per_unit * rr_target

    return {
        "entry": smart_round(price),
        "stop_loss": smart_round(stop),
        "take_profit": smart_round(target),
        "risk_reward": rr_target,
    }


# =============================================================================
# 7. СБОРКА ИТОГОВОГО SCORE
# =============================================================================
def score_ticker(last, patterns, ml_prob, ml_auc=None, daily_trend_up=None):
    score = 0.0
    direction = None

    if last["trend_up"]:
        score += 1.5
        direction = "long"
    elif last["trend_down"]:
        score += 1.5
        direction = "short"

    adx = last.get("ADX14", 0)
    trend_strength_mult = 1.3 if adx > 25 else (0.6 if adx < 20 else 1.0)

    if last["RSI14"] < 30:
        score += 1.0
        direction = direction or "long"
    elif last["RSI14"] > 70:
        score += 1.0
        direction = direction or "short"

    if last["MACD_hist"] > 0:
        score += 0.5
        direction = direction or "long"
    elif last["MACD_hist"] < 0:
        score += 0.5
        direction = direction or "short"

    stoch_k = last.get("STOCH_k", np.nan)
    if pd.notna(stoch_k):
        if stoch_k < 20:
            score += 0.4
            direction = direction or "long"
        elif stoch_k > 80:
            score += 0.4
            direction = direction or "short"

    willr = last.get("WILLR14", np.nan)
    if pd.notna(willr):
        if willr < -80:
            score += 0.3
            direction = direction or "long"
        elif willr > -20:
            score += 0.3
            direction = direction or "short"

    vwap_dev = last.get("price_vs_vwap_pct", 0)
    if pd.notna(vwap_dev):
        if vwap_dev > 0.3:
            score += 0.3
            direction = direction or "long"
        elif vwap_dev < -0.3:
            score += 0.3
            direction = direction or "short"

    bullish_kw = ["бычий", "вверх", "дно", "Перевёрнутая"]
    bearish_kw = ["медвежий", "вниз", "вершина"]
    for p in patterns:
        if any(k in p for k in bullish_kw):
            score += 1.3 if ("подтвержд" in p or "Пробой" in p) else 0.6
            direction = direction or "long"
        elif any(k in p for k in bearish_kw):
            score += 1.3 if ("подтвержд" in p or "Пробой" in p) else 0.6
            direction = direction or "short"

    if any("Бычья дивергенция" in p for p in patterns):
        score += 1.5
        direction = direction or "long"
    if any("Медвежья дивергенция" in p for p in patterns):
        score += 1.5
        direction = direction or "short"

    # Относительная сила к IMOEX
    rel_strength = last.get("rel_strength_pct", np.nan)
    if pd.notna(rel_strength):
        if rel_strength > 1.0:
            score += 0.5
            direction = direction or "long"
        elif rel_strength < -1.0:
            score += 0.5
            direction = direction or "short"

    score *= trend_strength_mult

    # Подтверждение по дневному тренду — сигнал ПО тренду старшего
    # таймфрейма получает бонус, ПРОТИВ тренда — штраф
    if daily_trend_up is not None and direction is not None:
        aligned = (daily_trend_up and direction == "long") or (not daily_trend_up and direction == "short")
        score *= 1.25 if aligned else 0.7

    if ml_prob is not None:
        if direction == "short":
            edge = (1 - ml_prob) - 0.5
        else:
            edge = ml_prob - 0.5
        auc_confidence = max(min((ml_auc - 0.5) * 10, 1.0), 0.0) if ml_auc else 0.3
        score += max(edge, 0) * 4 * auc_confidence

    if direction is None:
        direction = "long" if last["RSI14"] >= 50 else "short"

    return round(score, 2), direction


# =============================================================================
# ГЛАВНЫЙ PIPELINE
# =============================================================================
def run_screener(config=CONFIG):
    print("1/6: Быстрый префильтр по ликвидности...")
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(top_n)
    print(f"   Беру в работу {len(tickers)} тикеров")

    print("2/6: Скачиваю часовые свечи параллельно (без ещё не закрытого последнего бара)...")
    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )

    print("2b/6: Скачиваю индекс IMOEX для расчёта относительной силы...")
    try:
        index_df = get_index_candles("IMOEX", config["history_days"], config["candle_interval"])
    except Exception:
        index_df = None
        print("   Не удалось получить IMOEX — относительная сила будет пропущена")

    data_by_ticker, daily_trend_by_ticker = {}, {}
    skipped_low_vol, failed_indicator_tickers = 0, []
    for t, raw in raw_by_ticker.items():
        if len(raw) < 210:
            continue
        try:
            df_ind = add_indicators(raw)
            median_atr_pct = df_ind["ATR_pct"].tail(200).median()
            if median_atr_pct < config["min_atr_pct"]:
                skipped_low_vol += 1
                continue
            df_ind = compute_relative_strength(df_ind, index_df, config["rel_strength_window"])
            daily_trend_by_ticker[t] = add_daily_confirmation(raw)
            data_by_ticker[t] = df_ind
        except Exception as e:
            failed_indicator_tickers.append((t, str(e)))

    if failed_indicator_tickers:
        print(f"   Не удалось посчитать индикаторы для {len(failed_indicator_tickers)} тикеров "
              f"(напр. {failed_indicator_tickers[0][0]}: {failed_indicator_tickers[0][1][:80]}) — пропущены")

    print(f"   Хватает истории для анализа: {len(data_by_ticker)} тикеров "
          f"(отсеяно как низковолатильные фонды: {skipped_low_vol})")

    print("3/6: Обучаю ML-модель на объединённой истории всех тикеров...")
    model, auc = train_and_predict(
        data_by_ticker, config["ml_horizon_bars"], config["ml_up_threshold_atr"]
    )
    if auc is not None:
        print(f"   ROC-AUC на валидации (out-of-sample): {auc:.3f}")
        print("   (0.5 = случайность; веса ML в score уже приглушены пропорционально этому числу)")
    else:
        print("   Недостаточно данных для ML — работаем на техническом score.")

    print("4/6: Считаю свечные паттерны через pandas-ta-classic...")
    print("5/6: Считаю графические паттерны, дивергенции, риск-уровни и score...")
    results = []
    failed_result_tickers = []
    for t, df in data_by_ticker.items():
        try:
            candle_patterns = detect_candle_patterns(df)
            chart_patterns = detect_chart_patterns(df, config["pattern_lookback_bars"], config["swing_order"])
            divergences = detect_divergence(df, config["pattern_lookback_bars"], config["swing_order"])
            patterns = candle_patterns + chart_patterns + divergences
            last = df.iloc[-1]

            ml_prob = None
            if model is not None and not last[FEATURES].isna().any():
                ml_prob = float(model.predict_proba(last[FEATURES].values.reshape(1, -1))[0, 1])

            daily_trend_up = daily_trend_by_ticker.get(t)
            score, direction = score_ticker(last, patterns, ml_prob, ml_auc=auc, daily_trend_up=daily_trend_up)
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])
        except Exception as e:
            failed_result_tickers.append((t, str(e)))
            continue

        daily_trend_label = ("Совпадает" if daily_trend_up is not None and
                              ((daily_trend_up and direction == "long") or (not daily_trend_up and direction == "short"))
                              else ("Против" if daily_trend_up is not None else "Н/Д"))

        results.append({
            "Тикер": t,
            "Цена": smart_round(last["Close"]),
            "Направление": "ЛОНГ" if direction == "long" else "ШОРТ",
            "Score": score,
            "ML_проб_роста": round(ml_prob, 3) if ml_prob is not None else None,
            "RSI14": round(last["RSI14"], 1),
            "ADX14": round(last["ADX14"], 1),
            "Дневной тренд": daily_trend_label,
            "Отн.сила к IMOEX %": round(last["rel_strength_pct"], 2) if pd.notna(last.get("rel_strength_pct")) else None,
            "Паттерны": "; ".join(patterns) if patterns else "-",
            "Вход": risk["entry"],
            "Стоп-лосс": risk["stop_loss"],
            "Тейк-профит": risk["take_profit"],
            "Risk/Reward": risk["risk_reward"],
        })

    report = pd.DataFrame(results).sort_values("Score", ascending=False).reset_index(drop=True)

    if failed_result_tickers:
        print(f"   Пропущено на этапе паттернов/score: {len(failed_result_tickers)} тикеров "
              f"(напр. {failed_result_tickers[0][0]}: {failed_result_tickers[0][1][:80]})")

    print("6/6: Готово.\n")
    print("=" * 100)
    print(f"ТОП-{config['top_n_report']} ИДЕЙ ПО СКРИНЕРУ (не финансовая рекомендация!)")
    print("=" * 100)
    with pd.option_context("display.max_columns", None, "display.width", 220):
        print(report.head(config["top_n_report"]).to_string(index=False))

    return report


# =============================================================================
# ЗАПУСК
# =============================================================================
if __name__ == "__main__":
    full_report = run_screener(CONFIG)
    # full_report.to_csv("moex_screener_report.csv", index=False, encoding="utf-8-sig")

"""
ЧТО МОЖНО ДОБАВИТЬ ДАЛЬШЕ:
  - Полноценный бэктест самого score (не только ML) на исторических данных —
    единственный способ реально узнать, помогают ли веса в score_ticker()
    или нет, вместо того чтобы гадать по одному прогону.
  - Уровни Фибоначчи от последнего значимого свинга.
  - Ichimoku Cloud (есть в pandas-ta-classic: ta_input.ta.ichimoku()) —
    не включил по умолчанию, у него сложный многослойный вывод, стоит
    добавлять отдельно и тестировать на ваших тикерах.
  - Уведомления в Telegram, чтобы не заходить в Colab руками.
"""

1/6: Быстрый префильтр по ликвидности...
   Беру в работу 120 тикеров
2/6: Скачиваю часовые свечи параллельно (без ещё не закрытого последнего бара)...
   Из кэша сессии загружено сразу: 120 тикеров
2b/6: Скачиваю индекс IMOEX для расчёта относительной силы...
   Хватает истории для анализа: 103 тикеров (отсеяно как низковолатильные фонды: 17)
3/6: Обучаю ML-модель на объединённой истории всех тикеров...
   ROC-AUC на валидации (out-of-sample): 0.522
   (0.5 = случайность; веса ML в score уже приглушены пропорционально этому числу)
4/6: Считаю свечные паттерны через pandas-ta-classic...
5/6: Считаю графические паттерны, дивергенции, риск-уровни и score...
6/6: Готово.

ТОП-15 ИДЕЙ ПО СКРИНЕРУ (не финансовая рекомендация!)
       Тикер       Цена Направление  Score  ML_проб_роста  RSI14  ADX14 Дневной тренд  Отн.сила к IMOEX %                                                                                                                                                                   

'\nЧТО МОЖНО ДОБАВИТЬ ДАЛЬШЕ:\n  - Полноценный бэктест самого score (не только ML) на исторических данных —\n    единственный способ реально узнать, помогают ли веса в score_ticker()\n    или нет, вместо того чтобы гадать по одному прогону.\n  - Уровни Фибоначчи от последнего значимого свинга.\n  - Ichimoku Cloud (есть в pandas-ta-classic: ta_input.ta.ichimoku()) —\n    не включил по умолчанию, у него сложный многослойный вывод, стоит\n    добавлять отдельно и тестировать на ваших тикерах.\n  - Уведомления в Telegram, чтобы не заходить в Colab руками.\n'